# Adrenal Gland / Tumor Segmentation (nnU-Net v2)

How to run the released model on your own CT data.
Replace every `<...>` placeholder with a path on your machine.

| | |
|---|---|
| Task | `Dataset001_AdrenalTumor`, `3d_fullres`, `nnUNetTrainer__nnUNetPlans` |
| Input | single-channel CT, 1 mm isotropic, WW 400 / WL 40 |
| Trained on | **(128, 192, 192)** = (depth z, height y, width x) |
| Patch size | (96, 160, 160), sliding window |
| Labels | `0` background, `1` adrenal gland, `2` tumor |
| Folds | `fold_0` ... `fold_4` |

**1. Setup → 2. Preprocessing → 3. Inference → 4. Output**

## 1. Setup

```bash
pip install -e ./nnUNet-master
pip install -r requirements.txt
```

nnU-Net finds the weights through environment variables. Set them **before**
importing `nnunetv2`. Only `nnUNet_results` matters for inference.
([docs](https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/set_environment_variables.md))

In [ ]:
import os

REPO_ROOT = "<REPO_ROOT>"     # where this repository was cloned

os.environ["nnUNet_raw"]          = f"{REPO_ROOT}/nnUNet-master/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = f"{REPO_ROOT}/nnUNet-master/nnUNet_preprocessed"
os.environ["nnUNet_results"]      = f"{REPO_ROOT}/nnUNet-master/nnUNet_results"

# must exist and contain fold_0 ... fold_4
model_dir = (f"{os.environ['nnUNet_results']}/Dataset001_AdrenalTumor"
             f"/nnUNetTrainer__nnUNetPlans__3d_fullres")
print(sorted(os.listdir(model_dir)) if os.path.isdir(model_dir) else f"NOT FOUND: {model_dir}")

## 2. Preprocessing

Converts raw CT into the representation the model was trained on.

| Step | Operation |
|---|---|
| 1. Voxel normalization | resample to 1 x 1 x 1 mm (linear), clip HU to WW 400 / WL 40 |
| 2. In-plane resize | slice-wise center crop to 192 x 192 (pad first if smaller) |
| 3. Depth crop | keep 128 axial slices centered on the adrenal gland |

The crop center in step 3 is found automatically: a pre-trained ResNet-50
(`preprocess/VOI_extraction/Best.pth`) labels each slice adrenal present/absent, and the
middle of the longest positive run becomes the center.

Intensity normalization is **not** done here — nnU-Net applies its own `CTNormalization`
at inference time.

### Folder layout

Put your data under any `<YOUR_PROJECT_ROOT>`. `<CaseID>` can be any string; it carries
through to the predicted segmentation filename.

```
<YOUR_PROJECT_ROOT>/
├── data/<INSTITUTION>/<COHORT>/
│   ├── precontrast/<CaseID>.nii.gz          # <CaseID>/<CaseID>.nii.gz also works
│   └── postcontrast/<CaseID>.nii.gz
└── preprocessed/                            # everything below is written for you
    ├── <INSTITUTION>/<COHORT>/
    │   ├── <VERSION>/                       # FINAL volumes
    │   └── _intermediate/                   # logs, VOI csv, temp files — deletable
    └── nnunet_input/<INSTITUTION>_<COHORT>_<VERSION>/
                                             # <CaseID>_0000.nii.gz → nnUNetv2_predict -i
```

`version` = contrast phase: `precontrast`, `postcontrast`, or `total` (post if available,
else pre). `institution` and `disease` are **only folder names** — any label works
(`Adenoma`, `NF`, `ACC`, `CS`, ...) and nothing is looked up from them.

In [ ]:
import sys
sys.path.append(f"{REPO_ROOT}/preprocess")
from preprocess import *

In [ ]:
cfg = build_pre_config(
    institution="<YOUR_INSTITUTION>",     # folder name, e.g. "AMC"
    disease="<YOUR_COHORT>",              # folder name, e.g. "Adenoma" / "NF" / "ACC"
    version="precontrast",                # "precontrast" | "postcontrast" | "total"
    base_dir="<YOUR_PROJECT_ROOT>",       # holds ./data and ./preprocessed
    infer_gpu="0",                        # CUDA_VISIBLE_DEVICES for the VOI ResNet
    # target_size=192, target_depth=128,  # field of view — see "Field of view" below
    # checkpoint_path=f"{REPO_ROOT}/preprocess/VOI_extraction/Best.pth",   # default
)

print_config(cfg)   # resolved paths + warnings for anything missing

In [ ]:
# 1/5 resample to 1 mm, apply CT window, center crop to 192x192  -> _intermediate/resize_*
run_stage1(cfg)

# 2/5 split volumes into per-slice 2D .npy                       -> _intermediate/voi_npy_*
make_voi_npy(cfg)

# 3/5 ResNet-50: is an adrenal gland visible on this slice?      -> infer_result_*.csv
run_inference(cfg)

# 4/5 longest positive run per case -> center slice index        -> voi_info_*.csv
search_voi_index(cfg)

# 5/5 crop 128 slices around that center                         -> FINAL volumes
run_stage2_depthcrop(cfg)

# rename to <CaseID>_0000.nii.gz (required by nnU-Net) and return the folder to predict on
nnunet_input_dir = export_for_nnunet(cfg)
print(nnunet_input_dir)

In [ ]:
# Visual check: mid axial slice of each final volume.
# The adrenal region should be visible and the lesion should NOT touch the border.
qc_show(cfg)

# qc_show(cfg, show_range=(0, 10))   # first 10 cases only
# qc_show(cfg, which="resize")       # before the depth crop

### Field of view — important for large tumors

The 192 x 192 x 128 crop was chosen for **adrenal adenomas**, which are small. Large
lesions, adrenocortical carcinoma (ACC) in particular, can exceed it, and anything
outside the crop is discarded before the model ever sees it.

Check the `qc_show` images. If a lesion touches the border, enlarge the field of view:

```python
cfg = build_pre_config(..., target_size=320, target_depth=192)
```

or skip cropping entirely and keep the full CT:

```python
run_voxnorm_only(cfg)   # 1 mm resampling + windowing only
export_for_nnunet(cfg, out_dir="<YOUR_PROJECT_ROOT>/preprocessed/nnunet_input/full")
```

This costs inference time but no accuracy — nnU-Net accepts any input size (see §3).

### Other options

- **Everything at once:** `run_all(cfg)` (stage 1 → VOI → stage 2 → export).
- **From the shell:** `python preprocess/preprocess.py --base-dir <YOUR_PROJECT_ROOT> --institution <INST> --disease <COHORT> --version precontrast --all`
- **Your own VOI centers:** skip steps 2–4 and pass a CSV with columns `fname`,
  `voi_center_idx` → `run_stage2_depthcrop(cfg, voi_csv="<YOUR_PATH>/voi_info.csv")`
- **Failed detections:** `voi_center_idx = -1` means no adrenal slice was found; those
  cases are skipped. Fix the index manually in `_intermediate/voi_info_<version>.csv`.
- **Geometry:** output volumes have 1 mm spacing, zero origin, identity direction. The
  original scan's origin/direction are not preserved, so predictions cannot be overlaid
  on the *original* CT without re-registration.

## 3. Inference

Input files must be named **`<CaseID>_0000.nii.gz`** (`_0000` is the nnU-Net channel
suffix). `export_for_nnunet(cfg)` produces exactly that. Do not z-score the volumes
yourself — nnU-Net normalizes internally.

### Input size

The model was trained on **(128, 192, 192)** in numpy/SimpleITK axis order
`(z, y, x)` = (depth, height, width): 128 axial slices of 192 x 192 pixels. Read through
SimpleITK's image API, `GetSize()` reports the reverse, `(192, 192, 128)`.

**Your inputs do not have to match this.** nnU-Net runs sliding-window inference with a
(96, 160, 160) patch and stitches the result, so any shape works — smaller volumes are
padded automatically, larger ones just take more windows.

What *does* matter is **voxel spacing**: the plans file specifies 1 x 1 x 1 mm and nnU-Net
resamples anything else to it. The preprocessing above already produces that spacing, so
no extra interpolation is introduced.

In [ ]:
# -i    folder of <CaseID>_0000.nii.gz
# -o    output folder (created if missing)
# -d    dataset id 001 = Dataset001_AdrenalTumor
# -f    fold(s): `0` = single fold, `0 1 2 3 4` = ensemble of all five
# -c    configuration
# -chk  checkpoint_best.pth (best validation) or checkpoint_final.pth (last epoch)

!CUDA_VISIBLE_DEVICES=<GPU_ID> nnUNetv2_predict \
    -i <YOUR_PROJECT_ROOT>/preprocessed/nnunet_input/<INSTITUTION>_<COHORT>_<VERSION> \
    -o <YOUR_OUTPUT_PATH>/predictions \
    -d 001 -f 0 -c 3d_fullres \
    -chk checkpoint_best.pth

Useful extras: `--save_probabilities` (softmax `.npz`, only needed to ensemble across
runs), `-device cpu` (no GPU), `-npp 1 -nps 1` (less RAM), `--continue_prediction`
(resume).